This is auto loader job to load employee data from csv file from databricks volume to delta lake table in databricks. This job is scheduled to run every day at 12:00 PM.

In [2]:
print("Lets Begin with Execution of data loader notebook")

Lets Begin with Execution of data loader notebook


In [ ]:
main_catalog="workspace"
schema_name="works_dev"
volume_name="files"
target_table_name=main_catalog+"."+schema_name+".employees"

In [9]:
# 1. Define Volume and Metadata paths
# Replace with your actual catalog, schema, and volume names
base_volume_location = f"/Volumes/{main_catalog}/{schema_name}/{volume_name}/autoloaderfiles/data/"
base_schema_location = f"/Volumes/{main_catalog}/{schema_name}/{volume_name}/autoloaderfiles/metadata/schema/"
base_checkpoint_location = f"/Volumes/{main_catalog}/{schema_name}/{volume_name}/autoloaderfiles/metadata/checkpoint/"

In [10]:
print(f"Volume Path: {base_volume_location}")
print(f"Schema Location: {base_schema_location}")
print(f"Checkpoint Location: {base_checkpoint_location}")

Volume Path: /Volumes/workspace/works_dev/files/autoloaderfiles/data/
Schema Location: /Volumes/workspace/works_dev/files/autoloaderfiles/metadata/schema/
Checkpoint Location: /Volumes/workspace/works_dev/files/autoloaderfiles/metadata/checkpoint/


In [11]:
employee_data_location = base_volume_location + "employee/"
print(f"Employee CSV Path: {employee_data_location}")
employee_schema_location = base_schema_location + "employee/"   
print(f"Employee Schema Path: {employee_schema_location}")
employee_checkpoint_location = base_checkpoint_location + "employee/"
print(f"Employee Checkpoint Path: {employee_checkpoint_location}")


Employee CSV Path: /Volumes/workspace/works_dev/files/autoloaderfiles/data/employee/
Employee Schema Path: /Volumes/workspace/works_dev/files/autoloaderfiles/metadata/schema/employee/
Employee Checkpoint Path: /Volumes/workspace/works_dev/files/autoloaderfiles/metadata/checkpoint/employee/


In [ ]:
# 2. Read Employee CSVs with Auto Loader
# 'rescue' mode captures malformed rows or unexpected columns in _rescued_data
df_employees = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", employee_schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue") 
    .option("header", "true")              # Use the first line as headers
    .option("inferSchema", "true")         # Automatically detect data types (int, string, etc.)
    .load(employee_data_location))

In [ ]:
# 3. Write to a Delta Table in Unity Catalog
query = (df_employees.writeStream
    .format("delta")
    .option("checkpointLocation", employee_checkpoint_location)
    .option("mergeSchema", "true")          # Allow the target table schema to grow
    .trigger(availableNow=True)            # Process all new data and then stop
    .toTable(target_table_name))

ModuleNotFoundError: No module named 'pyspark'